In [1]:
#!/usr/bin/env python
# coding: utf-8
"""
═══════════════════════════════════════════════════════════════════════════════
 EvoAge — unified test / train / valid split builder
═══════════════════════════════════════════════════════════════════════════════

WHAT IT DOES (one pass over the data, fully vectorised)
  1. Build the global node-id mapping AND record each species' contiguous id
     block [start, end) — this is computed from the data, not hard-coded.
  2. Build the global relation-id mapping.
  3. Build one mapped-triples tensor per file-group (human + 5 species + species
     connection files), reusing the global mapping.
  4. Construct the test set in two parts, exactly mirroring the old pipeline:
       a. WITHIN-species 1 %  — 1 % of each species' own triples held out.
          • Human : random 1 % of all human triples (it is the base species).
          • Others: random 1 % drawn from edges whose BOTH endpoints live in
                    that species' own id block (species-specific nodes).
       b. CROSS-species (CS)  — edges linking a CONSERVED node to a species-
          specific node. Total CS count == Human-1 % count, split across the
          non-human species (see CS_SPLIT_MODE). This is the part that used the
          hard-coded `target_count = 248616`; it is now derived at run time.
  5. Everything not held out for test → pooled, deduplicated, and split 90/10
     into train / valid (seeded). Species-connection edges are train-only.
  6. Save tensors + DGL-KE-ready entities/relation .dict and train/valid/test
     .txt files.

KEY FIXES vs. the original notebooks
  • target_count NO LONGER hard-coded — derived from the realised Human-1 % size.
  • species id-block boundaries (the old 3062099 / 3258080 / … thresholds) are
    measured from the mapping build, not pasted in.
  • CS selection is RANDOM + seeded (old code took the first N matches → biased).
  • set(map(tuple, …)) difference  → bijective int64 encode + searchsorted.
  • python `int(h) in numpy_array` per-triple loop → torch.isin (vectorised).
  • node/relation mapping → parallel, column-projected, streamed (no 25 GB file
    ever fully in RAM). Pattern taken from EvoAge_1_to_1_1_to_M.py.
  • CS gene restriction auto-detects head_type/tail_type; falls back cleanly.

THINGS TO VERIFY BEFORE A REAL RUN  (search "VERIFY")
  • The path / glob block (SECTION 1) points at the files you actually want.
  • NON_HUMAN order matches the order genes were introduced (human files first).
  • SEED matches the rest of the EvoAge pipeline (you use 42 everywhere).
═══════════════════════════════════════════════════════════════════════════════
"""

import os
import glob
import warnings
from collections import OrderedDict
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
import pyarrow.parquet as pq
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════════
STORE          = "Store_House"
OUT_TAG        = "EvoAge_1to1_1toM"          # suffix used on every output file
SEED           = 42                          # VERIFY: matches the rest of EvoAge
WITHIN_FRAC    = 0.01                         # per-species held-out test fraction
VALID_FRAC     = 0.10                         # valid carved from the train pool
N_JOBS         = 8                            # mapping workers (tune to RAM)
BATCH_ROWS     = 2_000_000                    # rows per streamed batch
SAVE_TXT       = True                         # also dump DGL-KE .txt / .dict
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"   # isin/encode

# CS target: total cross-species test size and how it is divided.
#   "equal"        – total = Human-1% count, divided equally (original behaviour)
#   "per_species"  – each non-human species contributes 1% of ITS OWN size
#   "proportional" – total = Human-1% count, divided ∝ each species' size
CS_SPLIT_MODE  = "equal"

# CS edge definition: a conserved node linked to a species-specific node.
#   "auto" – use Gene restriction if head_type/tail_type exist, else conserved.
#   "gene" – force Gene restriction (errors if type columns are missing).
#   "conserved" – any human-block node ↔ species-block node (no type needed).
CS_EDGE_MODE   = "auto"

os.makedirs(STORE, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — FILE GROUPS                                              ★ VERIFY ★
#   groups MUST be ordered so that node ids are assigned human-first, then each
#   species. The contiguous id block per group is what makes within/CS work.
# ═══════════════════════════════════════════════════════════════════════════════
GENERALISED  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/"
SPECIES_ROOT = os.path.join(GENERALISED, "OTHER_SPECIES")
SPECIES_CONN = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/making_species_assocaitedwithconnection/one2one_plus_one2many_ortholog/"

NON_HUMAN = ["Yeast", "Celegans", "Drosophila", "Zebrafish", "Mouse"]   # introduction order
SPECIES_GLOB = "**/*1_to_one2one_plus_one2many.csv"                      # per-species ortholog files


def _clean(paths):
    return sorted(
        p for p in paths
        if "OTHER_SPECIES" not in p
        and ".ipynb_checkpoints" not in p
        and not os.path.basename(p).startswith("_")
        and not os.path.basename(p).endswith(".bak")
    )


def discover_groups():
    human = _clean(
        glob.glob(os.path.join(GENERALISED, "**/*.csv"), recursive=True)
        + glob.glob(os.path.join(GENERALISED, "**/*.parquet"), recursive=True)
    )
    groups = OrderedDict()
    groups["human"] = human
    for sp in NON_HUMAN:
        groups[sp.lower()] = sorted(
            f for f in glob.glob(os.path.join(SPECIES_ROOT, sp, SPECIES_GLOB), recursive=True)
            if ".ipynb_checkpoints" not in f
        )
    groups["species_conn"] = [
        os.path.join(SPECIES_CONN, f) for f in (
            "Homo_sapiens.parquet", "Saccharomyces_cerevisiae.parquet",
            "Caenorhabditis_elegans.parquet", "Drosophila_melanogaster.parquet",
            "Danio_rerio.parquet", "Mus_musculus.parquet",
        )
    ]
    for g, fs in groups.items():
        print(f"  {g:14s}: {len(fs)} files")
    return groups


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — PARALLEL MAPPING BUILDERS (node + relation) WITH BLOCK BOUNDARIES
#   Streamed, column-projected, parallel — copied from EvoAge_1_to_1_1_to_M.py and
#   extended to (a) also build the relation map and (b) record per-group id blocks.
# ═══════════════════════════════════════════════════════════════════════════════
HEAD, TAIL, RELATION = "head", "tail", "relation"


def _has(cols, names):
    s = {c.lower() for c in cols}
    return all(n in s for n in names)


def _file_unique(idx, path, col, want_pair):
    """Return (idx, sorted np.ndarray[str] uniques) for one column / column-pair."""
    seen = set()
    try:
        if path.lower().endswith(".parquet"):
            pf = pq.ParquetFile(path)
            names = pf.schema_arrow.names
            cols = [HEAD, TAIL] if want_pair else [col]
            if not _has(names, cols):
                return idx, None
            real = {n.lower(): n for n in names}
            read_cols = [real[c] for c in cols]
            for b in pf.iter_batches(columns=read_cols, batch_size=BATCH_ROWS):
                d = b.to_pandas(); d.columns = d.columns.str.lower()
                if want_pair:
                    seen.update(pd.unique(d[HEAD].astype(str)))
                    seen.update(pd.unique(d[TAIL].astype(str)))
                else:
                    seen.update(pd.unique(d[col].dropna().astype(str)))
        else:
            hdr = pd.read_csv(path, nrows=0).columns
            cols = [HEAD, TAIL] if want_pair else [col]
            if not _has(hdr, cols):
                return idx, None
            real = {c.lower(): c for c in hdr}
            read_cols = [real[c] for c in cols]
            for ch in pd.read_csv(path, usecols=read_cols, dtype=str,
                                  chunksize=BATCH_ROWS, low_memory=False):
                ch.columns = ch.columns.str.lower()
                if want_pair:
                    seen.update(pd.unique(ch[HEAD].astype(str)))
                    seen.update(pd.unique(ch[TAIL].astype(str)))
                else:
                    seen.update(pd.unique(ch[col].dropna().astype(str)))
    except Exception as e:
        print(f"  ❌ {os.path.basename(path)}: {e}")
        return idx, None
    if not seen:
        return idx, None
    arr = np.fromiter(seen, dtype=object, count=len(seen))
    arr.sort()                       # matches np.union1d's sorted order
    return idx, arr


def build_mapping_with_blocks(groups, want_pair=True, col=None, desc="mapping"):
    """
    Parallel phase-1 (per-file uniques) + ordered phase-2 (id assignment).
    Returns (mapping dict, blocks OrderedDict{group:(start,end)}).
    blocks is only meaningful for the node mapping (want_pair=True).
    """
    flat, group_of = [], []
    for g, fs in groups.items():
        for f in fs:
            flat.append(f); group_of.append(g)

    print(f"Phase 1 — {desc}: per-file uniques (parallel) …")
    results = [None] * len(flat)
    with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
        futs = {ex.submit(_file_unique, i, p, col, want_pair): i for i, p in enumerate(flat)}
        for fut in tqdm(as_completed(futs), total=len(futs), desc=desc, unit="file"):
            i, payload = fut.result()
            results[i] = payload

    print(f"Phase 2 — {desc}: assigning ids in file order …")
    mapping, n = {}, 0
    blocks, cur_group, start = OrderedDict(), None, 0
    for i, payload in enumerate(results):
        g = group_of[i]
        if g != cur_group:                       # group boundary
            if cur_group is not None:
                blocks[cur_group] = (start, n)
            cur_group, start = g, n
        if payload is None:
            continue
        for name in payload:                     # sorted within file
            if name not in mapping:
                mapping[name] = n; n += 1
    if cur_group is not None:
        blocks[cur_group] = (start, n)
    return mapping, blocks


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — PER-GROUP TENSOR BUILD  (vectorised .map(Series))
# ═══════════════════════════════════════════════════════════════════════════════
def _read_triples(path, type_cols):
    """Read head/tail/relation (+ head_type/tail_type if present & wanted)."""
    if path.lower().endswith(".parquet"):
        names = pq.ParquetFile(path).schema_arrow.names
    else:
        names = pd.read_csv(path, nrows=0).columns
    real = {c.lower(): c for c in names}
    want = ["head", "tail", "relation"] + (["head_type", "tail_type"] if type_cols else [])
    read = [real[w] for w in want if w in real]
    if path.lower().endswith(".parquet"):
        df = pd.read_parquet(path, columns=read)
    else:
        df = pd.read_csv(path, usecols=read, low_memory=False)
    df.columns = df.columns.str.lower()
    return df


def build_group_tensor(files, node_s, rel_s, want_type, end_human):
    """
    Map a group's files → triples tensor [N,3] (head, rel, tail).
    Also returns the set of CONSERVED node-ids that act as ortholog 'genes'
    for this group (human-block ids; restricted to Gene type if available).
    """
    parts, gene_ids = [], set()
    for path in tqdm(files, desc=f"  triples", unit="file", leave=False):
        try:
            df = _read_triples(path, want_type)
            if not {"head", "tail", "relation"}.issubset(df.columns):
                continue
            df = df.dropna(subset=["head", "tail", "relation"])
            if df.empty:
                continue
            h = df["head"].astype(str).map(node_s).fillna(-1).to_numpy("int64")
            t = df["tail"].astype(str).map(node_s).fillna(-1).to_numpy("int64")
            r = df["relation"].astype(str).map(rel_s).fillna(-1).to_numpy("int64")
            if (h < 0).any() or (t < 0).any() or (r < 0).any():
                tqdm.write(f"    ⚠️ unmapped values in {os.path.basename(path)}")
            parts.append(torch.from_numpy(np.stack([h, r, t], axis=1)))

            if want_type and {"head_type", "tail_type"}.issubset(df.columns):
                hg = h[(df["head_type"].astype(str).values == "Gene") & (h < end_human)]
                tg = t[(df["tail_type"].astype(str).values == "Gene") & (t < end_human)]
                gene_ids.update(hg.tolist()); gene_ids.update(tg.tolist())
        except Exception as e:
            tqdm.write(f"    ❌ {os.path.basename(path)}: {e}")
    if not parts:
        return torch.empty((0, 3), dtype=torch.int64), gene_ids
    return torch.cat(parts, 0), gene_ids


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — FAST TENSOR UTILITIES (bijective int64 packing)
# ═══════════════════════════════════════════════════════════════════════════════
def _bases(*tensors):
    """Per-column +1 maxima → bijective packing bases. Asserts int64 safety."""
    cm = None
    for t in tensors:
        if t.numel() == 0:
            continue
        m = t.to(torch.int64).amax(0)
        cm = m if cm is None else torch.maximum(cm, m)
    if cm is None:
        return 1, 1
    base_t, base_r = int(cm[2]) + 1, int(cm[1]) + 1
    assert int(cm[0]) * (base_r * base_t) + int(cm[1]) * base_t + int(cm[2]) < (1 << 63) - 1, \
        "int64 key overflow — fall back to torch.unique(dim=0)"
    return base_r, base_t


def encode(t, base_r, base_t):
    b = t.to(torch.int64)
    return b[:, 0] * (base_r * base_t) + b[:, 1] * base_t + b[:, 2]


def decode(keys, base_r, base_t):
    tail = keys % base_t
    rel = (keys // base_t) % base_r
    head = keys // (base_t * base_r)
    return torch.stack([head, rel, tail], 1)


def unique_triples(t, base_r, base_t):
    if t.numel() == 0:
        return t
    return decode(torch.unique(encode(t, base_r, base_t)), base_r, base_t)


def difference(A, B, base_r, base_t):
    """Rows of A not present in B (set difference on packed keys)."""
    if A.numel() == 0 or B.numel() == 0:
        return A
    ka = encode(A, base_r, base_t)
    kb, _ = torch.sort(encode(B, base_r, base_t))
    pos = torch.searchsorted(kb, ka).clamp_(max=kb.numel() - 1)
    return A[kb[pos] != ka]


def sample_rows(t, k, gen):
    """k random rows of t (without replacement); clamps k to len(t)."""
    k = min(int(k), t.shape[0])
    if k <= 0:
        return t[:0]
    idx = torch.randperm(t.shape[0], generator=gen)[:k]
    return t[idx]


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════
def main():
    gen = torch.Generator().manual_seed(SEED)

    print("\n■ SECTION 1 — discovering files")
    groups = discover_groups()

    print("\n■ SECTION 2a — node mapping + species id blocks")
    node_map, blocks = build_mapping_with_blocks(groups, want_pair=True, desc="nodes")
    print(f"  total nodes: {len(node_map):,}")
    for g, (s, e) in blocks.items():
        print(f"    {g:14s} id block [{s:>12,} , {e:>12,})   ({e - s:,} new nodes)")
    end_human = blocks["human"][1]

    print("\n■ SECTION 2b — relation mapping")
    rel_map, _ = build_mapping_with_blocks(groups, want_pair=False, col=RELATION, desc="relations")
    print(f"  total relations: {len(rel_map):,}")

    # Persist mappings (pickle + csv + DGL-KE dicts)
    node_df = pd.DataFrame({"Node": list(node_map), "MappedID": list(node_map.values())})
    rel_df = pd.DataFrame({"Relation": list(rel_map), "MappedID": list(rel_map.values())})
    node_df.to_pickle(f"{STORE}/node_id_{OUT_TAG}.pkl")
    rel_df.to_csv(f"{STORE}/relation_id_{OUT_TAG}.csv", index=False)

    node_s = pd.Series(node_map, dtype="int64")
    rel_s = pd.Series(rel_map, dtype="int64")

    # Decide CS edge mode from data
    sample_cols = pd.read_csv(
        next(f for f in groups[NON_HUMAN[0].lower()] if f.endswith(".csv")), nrows=0
    ).columns if groups[NON_HUMAN[0].lower()] else []
    have_types = _has(sample_cols, ["head_type", "tail_type"])
    if CS_EDGE_MODE == "gene" and not have_types:
        raise RuntimeError("CS_EDGE_MODE='gene' but head_type/tail_type not found.")
    use_gene = (CS_EDGE_MODE == "gene") or (CS_EDGE_MODE == "auto" and have_types)
    print(f"\n  CS edge mode: {'GENE-restricted' if use_gene else 'CONSERVED-node (human block)'}"
          f"  (type columns present: {have_types})")

    print("\n■ SECTION 3 — building per-group tensors")
    tensors, gene_sets = {}, {}
    for g, fs in groups.items():
        if not fs:
            tensors[g] = torch.empty((0, 3), dtype=torch.int64); gene_sets[g] = set(); continue
        print(f"  {g}")
        t, genes = build_group_tensor(fs, node_s, rel_s, use_gene, end_human)
        tensors[g], gene_sets[g] = t, genes
        print(f"    {t.shape[0]:,} triples")

    # global packing bases (over every triple we will touch)
    base_r, base_t = _bases(*tensors.values())

    print("\n■ SECTION 4 — within-species 1% test")
    within_tests, species_trains = {}, {}

    # Human: plain random 1% of all human triples (base species, no block filter)
    H = unique_triples(tensors["human"], base_r, base_t)
    k_h = int(WITHIN_FRAC * H.shape[0])
    test_h = sample_rows(H, k_h, gen)
    within_tests["human"] = test_h
    species_trains["human"] = difference(H, test_h, base_r, base_t)
    human_test_count = test_h.shape[0]
    print(f"  human: total {H.shape[0]:,} | test {human_test_count:,}")

    # Non-human: random 1% drawn from edges with BOTH endpoints in the species block
    for sp in NON_HUMAN:
        g = sp.lower()
        T = unique_triples(tensors[g], base_r, base_t)
        s, e = blocks[g]
        h_dev = T[:, 0].to(DEVICE); t_dev = T[:, 2].to(DEVICE)
        both_in = ((h_dev >= s) & (h_dev < e) & (t_dev >= s) & (t_dev < e)).cpu()
        pool = T[both_in]
        k = int(WITHIN_FRAC * T.shape[0])
        test_w = sample_rows(pool, k, gen)
        within_tests[g] = test_w
        species_trains[g] = difference(T, test_w, base_r, base_t)
        print(f"  {sp}: total {T.shape[0]:,} | eligible within-block {pool.shape[0]:,} | test {test_w.shape[0]:,}")

    print("\n■ SECTION 5 — cross-species test (dynamic target)")
    # ---- decide per-species CS targets ----
    sizes = {sp: tensors[sp.lower()].shape[0] for sp in NON_HUMAN}
    if CS_SPLIT_MODE == "per_species":
        cs_target = {sp: int(WITHIN_FRAC * sizes[sp]) for sp in NON_HUMAN}
    else:
        total = human_test_count
        if CS_SPLIT_MODE == "proportional":
            tot_sz = sum(sizes.values()) or 1
            cs_target = {sp: total * sizes[sp] // tot_sz for sp in NON_HUMAN}
        else:  # "equal"  (original behaviour)
            base = total // len(NON_HUMAN)
            cs_target = {sp: base for sp in NON_HUMAN}
        # hand the rounding remainder to the first species so the sum is exact
        rem = total - sum(cs_target.values())
        cs_target[NON_HUMAN[0]] += rem
    print(f"  CS targets ({CS_SPLIT_MODE}): "
          + ", ".join(f"{sp}={cs_target[sp]:,}" for sp in NON_HUMAN)
          + f"  | Σ={sum(cs_target.values()):,}  (Human-1% = {human_test_count:,})")

    cs_tests = {}
    for sp in NON_HUMAN:
        g = sp.lower()
        Tr = species_trains[g]                       # draw CS from the post-within train
        s, e = blocks[g]
        h = Tr[:, 0].to(DEVICE); t = Tr[:, 2].to(DEVICE)
        h_blk = (h >= s) & (h < e)
        t_blk = (t >= s) & (t < e)
        if use_gene:
            genes = torch.tensor(sorted(gene_sets[g]), dtype=torch.int64, device=DEVICE)
            h_cons = torch.isin(h, genes)
            t_cons = torch.isin(t, genes)
        else:
            h_cons = h < end_human
            t_cons = t < end_human
        cond = ((h_cons & t_blk) | (t_cons & h_blk)).cpu()
        pool = Tr[cond]
        test_cs = sample_rows(pool, cs_target[sp], gen)
        cs_tests[g] = test_cs
        species_trains[g] = difference(Tr, test_cs, base_r, base_t)
        print(f"  {sp}: CS-eligible {pool.shape[0]:,} | selected {test_cs.shape[0]:,}")

    print("\n■ SECTION 6 — assemble test / train / valid")
    test_all = unique_triples(
        torch.cat(list(within_tests.values()) + list(cs_tests.values()), 0), base_r, base_t)

    pool = torch.cat(list(species_trains.values()) + [tensors["species_conn"]], 0)
    pool = unique_triples(pool, base_r, base_t)
    pool = difference(pool, test_all, base_r, base_t)        # guarantee disjoint from test

    vmask = (torch.rand(pool.shape[0], generator=gen) < VALID_FRAC)
    valid = pool[vmask]
    train = pool[~vmask]

    print(f"  test  : {test_all.shape[0]:,}")
    print(f"  valid : {valid.shape[0]:,}")
    print(f"  train : {train.shape[0]:,}")
    print(f"  Σ     : {test_all.shape[0] + valid.shape[0] + train.shape[0]:,}")

    print("\n■ SECTION 7 — saving")
    torch.save(train,    f"{STORE}/{OUT_TAG}_train_90.pt")
    torch.save(valid,    f"{STORE}/{OUT_TAG}_valid_10.pt")
    torch.save(test_all, f"{STORE}/{OUT_TAG}_test.pt")
    # breakdown for auditing / the reviewer
    torch.save({k: v for k, v in within_tests.items()}, f"{STORE}/{OUT_TAG}_within_tests.pt")
    torch.save({k: v for k, v in cs_tests.items()},     f"{STORE}/{OUT_TAG}_cs_tests.pt")

    if SAVE_TXT:
        ent = node_df.copy()
        ent["Node"] = (ent["Node"].astype(str)
                       .str.replace(r"[\r\n]+", " ", regex=True)
                       .str.replace(r"\s+", " ", regex=True).str.strip())
        ent.sort_values("MappedID")[["MappedID", "Node"]].to_csv(
            f"{STORE}/entities_final.dict", sep="\t", index=False, header=False, encoding="utf-8")
        with open(f"{STORE}/relation_final.dict", "w", encoding="utf-8") as f:
            for name, idx in sorted(rel_map.items(), key=lambda kv: kv[1]):
                f.write(f"{idx}\t{name}\n")

        def write_tsv(t, path):
            pd.DataFrame(t.cpu().numpy()).to_csv(path, sep="\t", header=False, index=False)
        write_tsv(train,    f"{STORE}/{OUT_TAG}_train_90.txt")
        write_tsv(valid,    f"{STORE}/{OUT_TAG}_valid_10.txt")
        write_tsv(test_all, f"{STORE}/{OUT_TAG}_test.txt")

        # ── per-species WITHIN test files (Human.txt, Yeast.txt, …) ──────────
        # display names: "human" → "Human", non-human keep their NON_HUMAN label
        disp = {"human": "Human"}
        disp.update({sp.lower(): sp for sp in NON_HUMAN})
        test_dir = f"{STORE}/test_by_species"
        os.makedirs(test_dir, exist_ok=True)
        for g, t in within_tests.items():
            t = unique_triples(t, base_r, base_t)
            write_tsv(t, f"{test_dir}/{disp.get(g, g.capitalize())}.txt")
            print(f"    {disp.get(g, g):12s} within-test → {t.shape[0]:,}")

        # ── combined CROSS-SPECIES test file (CrossSpecies.txt) ──────────────
        if cs_tests:
            cs_all = unique_triples(torch.cat(list(cs_tests.values()), 0), base_r, base_t)
            write_tsv(cs_all, f"{test_dir}/CrossSpecies.txt")
            print(f"    {'CrossSpecies':12s} test        → {cs_all.shape[0]:,}")

    print("\n✅ done.")


if __name__ == "__main__":
    main()


■ SECTION 1 — discovering files
  human         : 84 files
  yeast         : 12 files
  celegans      : 24 files
  drosophila    : 20 files
  zebrafish     : 15 files
  mouse         : 23 files
  species_conn  : 6 files

■ SECTION 2a — node mapping + species id blocks
Phase 1 — nodes: per-file uniques (parallel) …


nodes:   0%|          | 0/184 [00:00<?, ?file/s]

Phase 2 — nodes: assigning ids in file order …
  total nodes: 45,555,533
    human          id block [           0 ,   45,289,069)   (45,289,069 new nodes)
    yeast          id block [  45,289,069 ,   45,338,152)   (49,083 new nodes)
    celegans       id block [  45,338,152 ,   45,372,575)   (34,423 new nodes)
    drosophila     id block [  45,372,575 ,   45,421,293)   (48,718 new nodes)
    zebrafish      id block [  45,421,293 ,   45,498,179)   (76,886 new nodes)
    mouse          id block [  45,498,179 ,   45,555,527)   (57,348 new nodes)
    species_conn   id block [  45,555,527 ,   45,555,533)   (6 new nodes)

■ SECTION 2b — relation mapping
Phase 1 — relations: per-file uniques (parallel) …


relations:   0%|          | 0/184 [00:00<?, ?file/s]

Phase 2 — relations: assigning ids in file order …
  total relations: 89

  CS edge mode: GENE-restricted  (type columns present: True)

■ SECTION 3 — building per-group tensors
  human


...

    27,524,671 triples
  mouse


  triples:   0%|          | 0/23 [00:00<?, ?file/s]

    36,151,527 triples
  species_conn


  triples:   0%|          | 0/6 [00:00<?, ?file/s]

    47,234,979 triples

■ SECTION 4 — within-species 1% test
  human: total 1,119,329,347 | test 11,193,293
  Yeast: total 6,409,734 | eligible within-block 1,560,446 | test 64,097
  Celegans: total 15,388,676 | eligible within-block 2,288,353 | test 153,886
  Drosophila: total 10,873,165 | eligible within-block 591,630 | test 108,731
  Zebrafish: total 24,047,369 | eligible within-block 382,043 | test 240,473
  Mouse: total 34,938,736 | eligible within-block 183,470 | test 183,470

■ SECTION 5 — cross-species test (dynamic target)
  CS targets (equal): Yeast=2,238,661, Celegans=2,238,658, Drosophila=2,238,658, Zebrafish=2,238,658, Mouse=2,238,658  | Σ=11,193,293  (Human-1% = 11,193,293)
  Yeast: CS-eligible 2,121,425 | selected 2,121,425
  Celegans: CS-eligible 2,152,080 | selected 2,152,080
  Drosophila: CS-eligible 957,275 | selected 957,275
  Zebrafish: CS-eligible 2,684,343 | selected 2,238,658
  Mouse: CS-eligible 2,349,963 | selected 2,238,658

■ SECTION 6 — assemble test / trai

In [2]:
11193293 + 11193293

22386586

In [ ]:
max_step = (10 * 1094670097) / 2048
max_step

In [3]:
!wc -l Store_House/*.txt

     153886 Store_House/Celegans.txt
    9708096 Store_House/CrossSpecies.txt
     108731 Store_House/Drosophila.txt
   21652046 Store_House/EvoAge_1to1_1toM_test.txt
 1094670097 Store_House/EvoAge_1to1_1toM_train_90.txt
  121621341 Store_House/EvoAge_1to1_1toM_valid_10.txt
   11193293 Store_House/Human.txt
     183470 Store_House/Mouse.txt
      64097 Store_House/Yeast.txt
     240473 Store_House/Zebrafish.txt
 1259595530 total


In [9]:
!ls Store_House  



Celegans.txt		       EvoAge_1to1_1toM_valid_10.txt
CrossSpecies.txt	       EvoAge_1to1_1toM_within_tests.pt
Drosophila.txt		       Human.txt
entities_final.dict	       Mouse.txt
EvoAge_1to1_1toM_cs_tests.pt   node_id_EvoAge_1to1_1toM.pkl
EvoAge_1to1_1toM_test.pt       relation_final.dict
EvoAge_1to1_1toM_test.txt      relation_id_EvoAge_1to1_1toM.csv
EvoAge_1to1_1toM_train_90.pt   test_by_species
EvoAge_1to1_1toM_train_90.txt  Yeast.txt
EvoAge_1to1_1toM_valid_10.pt   Zebrafish.txt


# Training and testing

In [10]:
from pathlib import Path
import pandas as pd

# Mapping your requested names to the actual filenames in Store_House
target_files = {
    "Human": "Human.txt",
    "Yeast": "Yeast.txt",
    "Celegans": "Celegans.txt",
    "Drosophila": "Drosophila.txt",
    "Zebrafish": "Zebrafish.txt",
    "Mouse": "Mouse.txt",
    "CrossSpecies": "CrossSpecies.txt",
}

directory = Path("Store_House")
data_list = []

for name, filename in target_files.items():
    file_path = directory / filename

    if file_path.exists():
        # Efficiently count lines without loading the file into RAM
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            line_count = sum(1 for _ in f)

        triple_count = line_count * 3

        # Append a dictionary for each row
        data_list.append(
            {
                "Species": name,
                "Line_Count": line_count,
                "Triple_Count": triple_count,
            }
        )
    else:
        print(f"Warning: {filename} not found in {directory}")

# Create the DataFrame
df = pd.DataFrame(data_list)
df

,Species,Line_Count,Triple_Count
0,Human,11193293,33579879
1,Yeast,64097,192291
2,Celegans,153886,461658
3,Drosophila,108731,326193
4,Zebrafish,240473,721419
5,Mouse,183470,550410
6,CrossSpecies,9708096,29124288


# Link Prediction
#### nohup bash Rescal_64_testing.sh > Rescal_64_testing.log &

In [12]:
#!/usr/bin/env python3
"""
Parse RESCAL dglke_eval test log file and extract results into CSV.
Automatically identifies species from test file names.
Adds Metric_type = LinkPrediction to each row.
"""

import re
import sys
from typing import Dict, List, Tuple

def parse_log_file(log_path: str) -> List[Dict]:
    """
    Parse log file and extract test results for each species.
    
    Looks for patterns:
    - "Reading test triples.... Finished. Read X test triples."
    - Test result section with MRR, HITS@1, HITS@3, HITS@10
    
    Maps results to species based on test set size (Total triples).
    """
    
    # Mapping of total triples to species name (from your data)
    SPECIES_MAPPING = {
        11193293: "Human",
        64097: "Yeast",
        153886: "Celegans",
        108731: "Drosophila",
        240473: "Zebrafish",
        183470: "Mouse",
        9708096: "CrossSpecies",
    }
    
    results = []
    
    with open(log_path, 'r') as f:
        content = f.read()
    
    # Split by "Reading test triples" to get individual test blocks
    blocks = content.split("Reading test triples")
    
    # Skip first block (it's before any test data)
    for block in blocks[1:]:
        try:
            # Extract test file size
            test_match = re.search(r"Finished\.\s+Read\s+(\d+)\s+test triples", block)
            if not test_match:
                continue
            total_triples = int(test_match.group(1))
            
            # Extract species name from data_files line (look backwards in original content)
            # Find MRR, HITS@1, HITS@3, HITS@10
            mrr_match = re.search(r"Test average MRR:\s+([\d.]+)", block)
            hits1_match = re.search(r"Test average HITS@1:\s+([\d.]+)", block)
            hits3_match = re.search(r"Test average HITS@3:\s+([\d.]+)", block)
            hits10_match = re.search(r"Test average HITS@10:\s+([\d.]+)", block)
            
            if not (mrr_match and hits1_match and hits3_match and hits10_match):
                continue
            
            mrr = float(mrr_match.group(1))
            hits1 = float(hits1_match.group(1))
            hits3 = float(hits3_match.group(1))
            hits10 = float(hits10_match.group(1))
            
            # Map by total triples count
            species = SPECIES_MAPPING.get(total_triples, f"Unknown_{len(results)}")
            
            results.append({
                "TestSet": species,
                "Metric_type": "LinkPrediction",
                "Total": total_triples,
                "MRR": mrr,
                "Hits@1": hits1,
                "Hits@3": hits3,
                "Hits@10": hits10,
            })
            
            print(f"✅ Parsed: {species} | Total={total_triples} | MRR={mrr:.4f}")
        
        except Exception as e:
            print(f"⚠️  Warning: Failed to parse block - {e}")
            continue
    
    return results

def write_csv(results: List[Dict], output_path: str):
    """Write results to CSV file."""
    
    if not results:
        print("❌ No results to write!")
        return False
    
    # Define column order
    columns = ["TestSet", "Metric_type", "Total", "MRR", "Hits@1", "Hits@3", "Hits@10"]
    
    # Write CSV
    with open(output_path, 'w') as f:
        # Header
        f.write(','.join(columns) + '\n')
        
        # Rows
        for result in results:
            row = [
                result["TestSet"],
                result["Metric_type"],
                str(result["Total"]),
                f"{result['MRR']:.4f}",
                f"{result['Hits@1']:.4f}",
                f"{result['Hits@3']:.4f}",
                f"{result['Hits@10']:.4f}",
            ]
            f.write(','.join(row) + '\n')
    
    return True

def main():
    # Configuration
    LOG_FILE = "/storage/Arushi/090526_EvoAge/kg_formation/training_2/1_per_species_test_EvoAge_121_12M/Rescal_64_testing.log"
    OUTPUT_CSV = "/storage/Arushi/090526_EvoAge/kg_formation/training_2/1_per_species_test_EvoAge_121_12M/rescal_1_percent_test_linkprediction_results.csv"
    
    print("=" * 80)
    print("📊 RESCAL Log Parser → CSV Converter")
    print("=" * 80 + "\n")
    
    print(f"📖 Reading log file: {LOG_FILE}\n")
    
    try:
        results = parse_log_file(LOG_FILE)
    except FileNotFoundError:
        print(f"❌ Error: Log file not found - {LOG_FILE}")
        return 1
    except Exception as e:
        print(f"❌ Error reading log: {e}")
        return 1
    
    if not results:
        print("❌ No test results found in log file!")
        return 1
    
    print(f"\n📝 Parsed {len(results)} test results\n")
    
    # Write CSV
    print(f"💾 Writing to CSV: {OUTPUT_CSV}\n")
    if not write_csv(results, OUTPUT_CSV):
        return 1
    
    # Display preview
    print("=" * 80)
    print("📋 PREVIEW OF OUTPUT CSV:")
    print("=" * 80)
    with open(OUTPUT_CSV, 'r') as f:
        for i, line in enumerate(f):
            print(line.rstrip())
            if i >= 8:  # Show header + all results
                break
    
    print("\n" + "=" * 80)
    print(f"✅ SUCCESS! CSV saved to:\n   {OUTPUT_CSV}\n")
    
    return 0

if __name__ == "__main__":
    sys.exit(main())

📊 RESCAL Log Parser → CSV Converter

📖 Reading log file: /storage/Arushi/090526_EvoAge/kg_formation/training_2/1_per_species_test_EvoAge_121_12M/Rescal_64_testing.log

✅ Parsed: Human | Total=11193293 | MRR=0.8701
✅ Parsed: Yeast | Total=64097 | MRR=0.9958
✅ Parsed: Celegans | Total=153886 | MRR=0.9976
✅ Parsed: Drosophila | Total=108731 | MRR=0.9796
✅ Parsed: Zebrafish | Total=240473 | MRR=0.9870
✅ Parsed: Mouse | Total=183470 | MRR=0.8921
✅ Parsed: CrossSpecies | Total=9708096 | MRR=0.9904

📝 Parsed 7 test results

💾 Writing to CSV: /storage/Arushi/090526_EvoAge/kg_formation/training_2/1_per_species_test_EvoAge_121_12M/rescal_1_percent_test_linkprediction_results.csv

📋 PREVIEW OF OUTPUT CSV:
TestSet,Metric_type,Total,MRR,Hits@1,Hits@3,Hits@10
Human,LinkPrediction,11193293,0.8701,0.8026,0.9217,0.9975
Yeast,LinkPrediction,64097,0.9958,0.9920,0.9995,0.9997
Celegans,LinkPrediction,153886,0.9976,0.9965,0.9985,0.9992
Drosophila,LinkPrediction,108731,0.9796,0.9689,0.9883,0.9958
Zebrafish,L

SystemExit: 0

In [16]:
Link_pred = pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/training_2/1_per_species_test_EvoAge_121_12M/rescal_1_percent_test_linkprediction_results.csv')
Link_pred

,TestSet,Metric_type,Total,MRR,Hits@1,Hits@3,Hits@10
0,Human,LinkPrediction,11193293,0.8701,0.8026,0.9217,0.9975
1,Yeast,LinkPrediction,64097,0.9958,0.9920,0.9995,0.9997
2,Celegans,LinkPrediction,153886,0.9976,0.9965,0.9985,0.9992
3,Drosophila,LinkPrediction,108731,0.9796,0.9689,0.9883,0.9958
4,Zebrafish,LinkPrediction,240473,0.9870,0.9805,0.9918,0.9966
5,Mouse,LinkPrediction,183470,0.8921,0.8656,0.9013,0.9373
6,CrossSpecies,LinkPrediction,9708096,0.9904,0.9864,0.9934,0.9952


# Edge type prediction 

#### ! python 1_per_rescal_edge_type_metrics.py 

In [15]:
Edge_type = pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/training_2/1_per_species_test_EvoAge_121_12M/rescal_1_percent_test_edgeType_results.csv')
Edge_type

,TestSet,Metric_type,Total,MRR,Hits@1,Hits@3,Hits@10
0,Celegans,EdgeType,153886,0.9585,0.9409,0.9718,0.9869
1,CrossSpecies,EdgeType,9708096,0.7766,0.7017,0.8269,0.8969
2,Drosophila,EdgeType,108731,0.8468,0.7984,0.8776,0.9404
3,Human,EdgeType,11193293,0.6617,0.5874,0.6907,0.8108
4,Mouse,EdgeType,183470,0.4499,0.3569,0.4785,0.6336
5,Yeast,EdgeType,64097,0.9817,0.9755,0.9856,0.9938
6,Zebrafish,EdgeType,240473,0.8692,0.8277,0.8949,0.9488
